# CrewAI Literature Annotation Notebook

This notebook runs the multi-agent CrewAI pipeline to annotate a gene using literature from PubMed (MongoDB) and the Reactome graph (Neo4j).

**Requirements before running:**
- MongoDB must be running with PubMed abstracts loaded (`persist_abstracts_to_mongo()` complete)
- Neo4j must be running with the Reactome graph loaded
- `ANTHROPIC_API_KEY` must be set in `.env`

In [ ]:
import asyncio
import json
import os
import sys
from pathlib import Path

# The tool code resolves resource paths relative to the current working directory
# (e.g. 'resources/reactome_domain_model.json', 'resources/interactions/...',
# 'data/papers/...'). This notebook lives in notebooks/, so move the working
# directory up to the repo root once, here, or none of those paths resolve.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

# Make reactome_llm importable
sys.path.append(str(REPO_ROOT / 'reactome_llm'))

from GenePathwayAnnotator import GenePathwayAnnotator
from CrewAILiteratureAnnotator import CrewAILiteratureAnnotator, AnnotationRequest
from ModelConfig import create_reactome_chat_model, get_crewai_model_settings

assert os.getenv('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not loaded — check .env at the repo root'
print(f'Imports OK; env loaded; cwd = {Path.cwd()}')

## Set your gene here

Change `GENE` to whatever gene you want to annotate.

In [ ]:
GENE = "SHANK3"  # <-- change this to any gene you want to annotate

MAX_PAPERS = 5       # max papers to retrieve from MongoDB
QUALITY_THRESHOLD = 0.7  # minimum quality score to accept annotation

print(f'Gene: {GENE}')

## Initialize the annotator

In [ ]:
base_model = create_reactome_chat_model()
crewai_model_name, crewai_temperature = get_crewai_model_settings()

annotator = GenePathwayAnnotator()
annotator.set_model(base_model)

crewai_annotator = CrewAILiteratureAnnotator(
    annotator,
    model=crewai_model_name,
    temperature=crewai_temperature,
    max_iter=3,
    verbose=True
)

print('Annotator initialized')

## Run the annotation

This uses `enable_literature_search=True` so the pipeline searches MongoDB for relevant papers automatically — no need to provide specific PMIDs.

In [ ]:
request = AnnotationRequest(
    gene=GENE,
    max_papers=MAX_PAPERS,
    quality_threshold=QUALITY_THRESHOLD,
    enable_full_text=False,
    enable_literature_search=True  # searches MongoDB automatically
)

result = await crewai_annotator.annotate_literature(request)
print(f'Annotation complete for {result.gene}')

## Inspect results

In [ ]:
print(f"Gene: {result.gene}")
print(f"Reactome instances created: {len(result.reactome_instances)}")
print(f"Literature evidence: {len(result.literature_evidence)}")
print(f"Quality scores: {result.quality_scores}")
print(f"Validation status: {result.validation_report.get('approval_status', 'N/A')}")

In [ ]:
# View the full Reactome instances output
print(json.dumps(result.reactome_instances, indent=2))

## Export results to file

In [ ]:
output_path = f"results/annotation_{result.gene}.json"
Path('results').mkdir(exist_ok=True)
crewai_annotator.export_results(result, output_path)
print(f"Results saved to {output_path}")